# Swin Transformer — UAV Person / Non-Person Classification

This Google Colab notebook fine-tunes the official **Swin-Tiny (`microsoft/swin-tiny-patch4-window7-224`)** ImageNet-1K model for two classes: `Non-Person` and `Person`. The official model is trained at 224×224 and has about 28.3M parameters. citeturn0search0

**Important:** `swin_person.pth` is a custom checkpoint produced by this training process. The original Swin checkpoint is not specifically trained for UAV person recognition. citeturn0search0turn0search6

Dataset structure:

```text
dataset/
├── person/
│   ├── person001.jpg
│   └── ...
└── non_person/
    ├── empty001.jpg
    └── ...
```

Use many varied UAV images in both classes. The supplied `uav_person.jpg` is only a demonstration image and is not sufficient to train a reliable classifier.

In [10]:
!pip -q install transformers accelerate scikit-learn pillow matplotlib

In [2]:
import os, random, zipfile
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader, random_split
from PIL import Image
from sklearn.metrics import classification_report, confusion_matrix
from transformers import AutoImageProcessor, AutoModelForImageClassification

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
MODEL_NAME = "microsoft/swin-tiny-patch4-window7-224"
DATA_DIR = "/content"
MODEL_OUT = "/content/swin_person.pth"
BATCH_SIZE = 8
EPOCHS = 8
LR = 2e-5
WEIGHT_DECAY = 0.01

print("Device:", DEVICE)
print("Model:", MODEL_NAME)

Device: cpu
Model: microsoft/swin-tiny-patch4-window7-224


In [6]:
# Upload a ZIP containing dataset/person and dataset/non_person
from google.colab import files

uploaded = files.upload()
for name in uploaded:
    if name.lower().endswith(".zip"):
        with zipfile.ZipFile(name, "r") as z:
            z.extractall("/content")
        print("Extracted:", name)

print("Dataset exists:", os.path.exists(DATA_DIR))

Saving simple_person_nonperson_dataset.zip to simple_person_nonperson_dataset.zip
Extracted: simple_person_nonperson_dataset.zip
Dataset exists: True


In [7]:
assert os.path.isdir(os.path.join(DATA_DIR, "person")), "Missing dataset/person folder"
assert os.path.isdir(os.path.join(DATA_DIR, "non_person")), "Missing dataset/non_person folder"

for cls in ["person", "non_person"]:
    n = len([
        f for f in os.listdir(os.path.join(DATA_DIR, cls))
        if f.lower().endswith((".jpg", ".jpeg", ".png", ".bmp", ".webp"))
    ])
    print(cls, ":", n, "images")

person : 20 images
non_person : 20 images


In [8]:
processor = AutoImageProcessor.from_pretrained(MODEL_NAME)

class UAVDataset(Dataset):
    def __init__(self, root, processor):
        self.processor = processor
        self.samples = []
        for folder, label in [("non_person", 0), ("person", 1)]:
            folder_path = os.path.join(root, folder)
            for fname in sorted(os.listdir(folder_path)):
                if fname.lower().endswith((".jpg", ".jpeg", ".png", ".bmp", ".webp")):
                    self.samples.append((os.path.join(folder_path, fname), label))

        if not self.samples:
            raise RuntimeError("No images found.")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        image = Image.open(path).convert("RGB")
        pixel_values = self.processor(
            images=image, return_tensors="pt"
        )["pixel_values"].squeeze(0)
        return pixel_values, torch.tensor(label, dtype=torch.long)

dataset = UAVDataset(DATA_DIR, processor)

n_total = len(dataset)
n_train = int(0.8 * n_total)
n_val = n_total - n_train

generator = torch.Generator().manual_seed(SEED)
train_ds, val_ds = random_split(dataset, [n_train, n_val], generator=generator)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

print("Total:", n_total, "Train:", n_train, "Validation:", n_val)

preprocessor_config.json:   0%|          | 0.00/255 [00:00<?, ?B/s]

Total: 40 Train: 32 Validation: 8


In [9]:
id2label = {0: "Non-Person", 1: "Person"}
label2id = {"Non-Person": 0, "Person": 1}

model = AutoModelForImageClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2,
    id2label=id2label,
    label2id=label2id,
    ignore_mismatched_sizes=True
).to(DEVICE)

criterion = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(
    model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY
)

print(
    "Trainable parameters:",
    sum(p.numel() for p in model.parameters() if p.requires_grad)
)

config.json:   0%|          | 0.00/71.8k [00:00<?, ?B/s]

[transformers] You passed `num_labels=2` which is incompatible to the `id2label` map of length `1000`.


model.safetensors: reconstructing file:   0%|          |  0.00B /  113MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/221 [00:00<?, ?it/s]

[transformers] SwinForImageClassification LOAD REPORT from: microsoft/swin-tiny-patch4-window7-224
Key               | Status   |                                                                                          
------------------+----------+------------------------------------------------------------------------------------------
classifier.weight | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([1000, 768]) vs model:torch.Size([2, 768])
classifier.bias   | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([1000]) vs model:torch.Size([2])          

Notes:
- MISMATCH:	ckpt weights were loaded, but they did not match the original empty weight shapes.


Trainable parameters: 27520892


In [10]:
def evaluate(model, loader):
    model.eval()
    total_loss = 0.0
    y_true, y_pred = [], []

    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            logits = model(pixel_values=x).logits
            loss = criterion(logits, y)

            total_loss += loss.item() * x.size(0)
            y_true.extend(y.cpu().numpy().tolist())
            y_pred.extend(logits.argmax(dim=1).cpu().numpy().tolist())

    acc = np.mean(np.array(y_true) == np.array(y_pred))
    return total_loss / len(loader.dataset), acc, y_true, y_pred

best_val_acc = -1.0

for epoch in range(EPOCHS):
    model.train()
    running = 0.0

    for x, y in train_loader:
        x, y = x.to(DEVICE), y.to(DEVICE)

        optimizer.zero_grad()
        logits = model(pixel_values=x).logits
        loss = criterion(logits, y)
        loss.backward()
        optimizer.step()

        running += loss.item() * x.size(0)

    train_loss = running / len(train_loader.dataset)
    val_loss, val_acc, _, _ = evaluate(model, val_loader)

    print(
        f"Epoch {epoch+1:02d}/{EPOCHS} | "
        f"train_loss={train_loss:.4f} | "
        f"val_loss={val_loss:.4f} | "
        f"val_acc={val_acc:.4f}"
    )

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save({
            "model_state_dict": model.state_dict(),
            "model_name": MODEL_NAME,
            "id2label": id2label,
            "label2id": label2id,
            "best_val_accuracy": best_val_acc
        }, MODEL_OUT)
        print("Saved:", MODEL_OUT)

Epoch 01/8 | train_loss=0.5115 | val_loss=0.2065 | val_acc=1.0000
Saved: /content/swin_person.pth
Epoch 02/8 | train_loss=0.1548 | val_loss=0.0591 | val_acc=1.0000
Epoch 03/8 | train_loss=0.0466 | val_loss=0.0153 | val_acc=1.0000
Epoch 04/8 | train_loss=0.0213 | val_loss=0.0043 | val_acc=1.0000
Epoch 05/8 | train_loss=0.0066 | val_loss=0.0017 | val_acc=1.0000
Epoch 06/8 | train_loss=0.0037 | val_loss=0.0008 | val_acc=1.0000
Epoch 07/8 | train_loss=0.0012 | val_loss=0.0004 | val_acc=1.0000
Epoch 08/8 | train_loss=0.0014 | val_loss=0.0003 | val_acc=1.0000


In [11]:
checkpoint = torch.load(MODEL_OUT, map_location=DEVICE, weights_only=False)
model.load_state_dict(checkpoint["model_state_dict"])

val_loss, val_acc, y_true, y_pred = evaluate(model, val_loader)

print("Best validation accuracy:", checkpoint["best_val_accuracy"])
print("\nClassification report:")
print(
    classification_report(
        y_true,
        y_pred,
        target_names=["Non-Person", "Person"],
        digits=4
    )
)
print("Confusion matrix:\n", confusion_matrix(y_true, y_pred))

Best validation accuracy: 1.0

Classification report:
              precision    recall  f1-score   support

  Non-Person     1.0000    1.0000    1.0000         2
      Person     1.0000    1.0000    1.0000         6

    accuracy                         1.0000         8
   macro avg     1.0000    1.0000    1.0000         8
weighted avg     1.0000    1.0000    1.0000         8

Confusion matrix:
 [[2 0]
 [0 6]]


In [16]:
# Upload uav_person.jpg and run inference
from google.colab import files

uploaded = files.upload()
test_path = next(iter(uploaded.keys()))

image = Image.open(test_path).convert("RGB")
inputs = processor(images=image, return_tensors="pt")
pixel_values = inputs["pixel_values"].to(DEVICE)

model.eval()
with torch.no_grad():
    logits = model(pixel_values=pixel_values).logits
    probs = torch.softmax(logits, dim=1)[0]

pred = int(torch.argmax(probs).item())

print("Image:", test_path)
print("Prediction:", id2label[pred])
print(f"Confidence: {probs[pred].item()*100:.2f}%")
print(f"Non-Person: {probs[0].item()*100:.2f}%")
print(f"Person: {probs[1].item()*100:.2f}%")

Saving person_16.jpg to person_16.jpg
Image: person_16.jpg
Prediction: Person
Confidence: 81.71%
Non-Person: 18.29%
Person: 81.71%


In [33]:
# Download the trained UAV person classifier
from google.colab import files
files.download(MODEL_OUT)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## What the resulting model does

`sw​in_person.pth` is a **binary image classifier**. It predicts:

- `Person`
- `Non-Person`

and reports the softmax confidence.

It **does not produce a bounding box**. For UAV person localization, use an object-detection model and evaluate with precision, recall, mAP and IoU.